In [1]:
import os
from pyspark.sql import SparkSession, functions as F

# Имя каталога
catalog = "lk"

# Доступ к minio
access_key = os.getenv("MINIO_ROOT_USER", "minioadmin")
secret_key = os.getenv("MINIO_ROOT_PASSWORD", "minioadmin")
warehouse = os.getenv("LAKEKEEPER_WAREHOUSE", "mydatalab")

#Настройка каталога в Spark
spark = (
    SparkSession.builder.appName("lakekeeper-iceberg-demo")
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config(f"spark.sql.catalog.{catalog}", "org.apache.iceberg.spark.SparkCatalog")
    .config(f"spark.sql.catalog.{catalog}.type", "rest")
    .config(f"spark.sql.catalog.{catalog}.uri", "http://127.0.0.1:8181/catalog")
    .config(f"spark.sql.catalog.{catalog}.warehouse", warehouse)
    .config(f"spark.sql.catalog.{catalog}.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
    .config(f"spark.sql.catalog.{catalog}.s3.endpoint", "http://127.0.0.1:9000")
    .config(f"spark.sql.catalog.{catalog}.s3.path-style-access", "true")
    .config(f"spark.sql.catalog.{catalog}.s3.access-key-id", access_key)
    .config(f"spark.sql.catalog.{catalog}.s3.secret-access-key", secret_key)
    .config(f"spark.sql.legacy.parquet.nanosAsLong","true") #Добавили для поддержки Timestamp из паркета
    .config("spark.sql.defaultCatalog", catalog)
    .getOrCreate()
)

#Уровень логирования
spark.sparkContext.setLogLevel("WARN")

In [2]:
# Добавления поля во все таблицы
spark.sql("ALTER TABLE lk.stage.users ADD COLUMN last_updated TIMESTAMP")
spark.sql("ALTER TABLE lk.stage.payments ADD COLUMN last_updated TIMESTAMP")
spark.sql("ALTER TABLE lk.stage.trips ADD COLUMN last_updated TIMESTAMP")
spark.sql("ALTER TABLE lk.stage.events ADD COLUMN last_updated TIMESTAMP")

DataFrame[]

In [4]:
# Обновление поля во всех таблицах
spark.sql("UPDATE TABLE lk.stage.users set last_updated=current_timestamp()")
spark.sql("UPDATE TABLE lk.stage.payments set last_updated=current_timestamp()")
spark.sql("UPDATE TABLE lk.stage.trips set last_updated=current_timestamp()")
spark.sql("UPDATE TABLE lk.stage.events set last_updated=current_timestamp()")

ParseException: 
[PARSE_SYNTAX_ERROR] Syntax error at or near '.'. SQLSTATE: 42601 (line 1, pos 15)

== SQL ==
UPDATE TABLE lk.stage.users set last_updated=current_timestamp()
---------------^^^
